In [2]:
import pandas as pd

# Load the cleaned customer-level dataset created in Phase 1.
# We use the processed file instead of the raw file because:
# - the "Surburban" typo has already been fixed
# - our derived purchase_preference feature is already included
# - the original raw dataset stays untouched

df = pd.read_csv("../data/processed/hillstrom_clean.csv")

# Confirm the dataset loaded correctly.
print(df.shape)
df.head()

(64000, 13)


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend,purchase_preference
0,10,2) $100 - $200,142.44,1,0,Suburban,0,Phone,Womens E-Mail,0,0,0.0,Mens Only
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0,Both
2,7,2) $100 - $200,180.65,0,1,Suburban,1,Web,Womens E-Mail,0,0,0.0,Womens Only
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0,Mens Only
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0,Mens Only


In [3]:
# -------------------------
# Define features and targets
# -------------------------
#
# IMPORTANT:
# We only use customer information that existed BEFORE the email was sent.
#
# We do NOT use:
# - visit
# - conversion
# - spend
#
# as input features because those happen AFTER treatment.

feature_columns = [
    "recency",
    "history",
    "mens",
    "womens",
    "zip_code",
    "newbie",
    "channel",
    "purchase_preference"
]

# Predictive target 1:
# Did the customer convert?
conversion_target = "conversion"

# Predictive target 2:
# How much did the customer spend?
spend_target = "spend"

print("Features:")
print(feature_columns)

print("\nTargets:")
print(conversion_target, spend_target)

Features:
['recency', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'purchase_preference']

Targets:
conversion spend


In [4]:
# -------------------------
# Check conversion class balance
# -------------------------
#
# conversion = 1 means the customer purchased.
# conversion = 0 means the customer did not purchase.
#
# Before training a classification model, we need to know
# how common each outcome is.

conversion_counts = df["conversion"].value_counts()

conversion_rate = df["conversion"].mean()

print("Conversion counts:")
print(conversion_counts)

print("\nOverall conversion rate:")
print(f"{conversion_rate:.2%}")

Conversion counts:
conversion
0    63422
1      578
Name: count, dtype: int64

Overall conversion rate:
0.90%


In [5]:
from sklearn.model_selection import train_test_split

# -------------------------
# Create features (X) and target (y)
# -------------------------

# X contains only pre-treatment customer information.
X = df[feature_columns].copy()

# y is the binary outcome we want to predict:
# 1 = converted
# 0 = did not convert
y = df["conversion"].copy()


# -------------------------
# Split into training and test sets
# -------------------------
#
# test_size=0.20 means:
# - 80% of customers are used to train the model
# - 20% are held back for final evaluation
#
# stratify=y keeps the rare conversion rate similar
# in both train and test sets.
#
# random_state=42 makes the split reproducible.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])

print("\nTraining conversion rate:")
print(f"{y_train.mean():.2%}")

print("\nTest conversion rate:")
print(f"{y_test.mean():.2%}")

Training rows: 51200
Test rows: 12800

Training conversion rate:
0.90%

Test conversion rate:
0.91%


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# -------------------------
# Separate numeric and categorical features
# -------------------------

numeric_features = [
    "recency",
    "history",
    "mens",
    "womens",
    "newbie"
]

categorical_features = [
    "zip_code",
    "channel",
    "purchase_preference"
]


# -------------------------
# Build preprocessing logic
# -------------------------
#
# Numeric features can pass through unchanged.
#
# Categorical features need one-hot encoding.
# Example:
#
# channel = Web
# becomes something like:
#
# channel_Web = 1
# channel_Phone = 0
# channel_Multichannel = 0

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [7]:
from sklearn.linear_model import LogisticRegression

# -------------------------
# Build the baseline model pipeline
# -------------------------
#
# A Pipeline connects:
#
# raw customer data
#      ↓
# preprocessing
#      ↓
# logistic regression model

conversion_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

print("Baseline conversion model created.")

Baseline conversion model created.


In [8]:
# Training the baseline model


conversion_model.fit(
    X_train,
    y_train
)

print("Baseline conversion model trained.")

Baseline conversion model trained.


In [9]:
# -------------------------
# Predict on the test set
# -------------------------
#
# predict() gives a hard class:
# 0 = no conversion
# 1 = conversion
#
# predict_proba() gives the model's estimated probability
# that each customer converts.
#
# We will use probabilities later because they are much more useful
# than only saying yes/no.

y_pred = conversion_model.predict(X_test)

y_prob = conversion_model.predict_proba(X_test)[:, 1]

print("Predictions created.")
print("Number of test predictions:", len(y_pred))

Predictions created.
Number of test predictions: 12800


In [10]:
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# -------------------------
# Evaluate the baseline conversion model
# -------------------------
#
# ROC-AUC:
# How well does the model rank converters above non-converters?
#
# Average Precision / PR-AUC:
# Especially useful when the positive class is rare.
#
# Confusion matrix:
# Shows how many converters/non-converters we classified correctly or incorrectly.

roc_auc = roc_auc_score(y_test, y_prob)

pr_auc = average_precision_score(y_test, y_prob)

print("ROC-AUC:", roc_auc)
print("PR-AUC:", pr_auc)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

ROC-AUC: 0.6215201883448058
PR-AUC: 0.017796938874125506

Confusion Matrix:
[[7603 5081]
 [  49   67]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9936    0.5994    0.7477     12684
           1     0.0130    0.5776    0.0255       116

    accuracy                         0.5992     12800
   macro avg     0.5033    0.5885    0.3866     12800
weighted avg     0.9847    0.5992    0.7412     12800



In [11]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# -------------------------
# Build a no-skill baseline
# -------------------------
#
# DummyClassifier does not really learn customer patterns.
#
# strategy="prior" means it simply predicts using the overall
# conversion rate found in the training data.
#
# We use this as a floor:
# our real model should beat this.

dummy_model = DummyClassifier(
    strategy="prior",
    random_state=42
)

# Train the dummy model.
dummy_model.fit(X_train, y_train)

# Get conversion probabilities on the test set.
dummy_prob = dummy_model.predict_proba(X_test)[:, 1]

# Evaluate it with the same metrics as our logistic regression.
dummy_roc_auc = roc_auc_score(
    y_test,
    dummy_prob
)

dummy_pr_auc = average_precision_score(
    y_test,
    dummy_prob
)

print("Dummy ROC-AUC:", dummy_roc_auc)
print("Dummy PR-AUC:", dummy_pr_auc)

print("\nLogistic Regression ROC-AUC:", roc_auc)
print("Logistic Regression PR-AUC:", pr_auc)

Dummy ROC-AUC: 0.5
Dummy PR-AUC: 0.0090625

Logistic Regression ROC-AUC: 0.6215201883448058
Logistic Regression PR-AUC: 0.017796938874125506


In [12]:
from sklearn.metrics import precision_score, recall_score, f1_score

# -------------------------
# Compare different probability thresholds
# -------------------------
#
# The model gives each customer a probability of conversion.
#
# Example:
# 0.70 = model thinks the customer has a 70% chance
# 0.20 = model thinks the customer has a 20% chance
#
# To turn probability into a yes/no prediction,
# we need a cutoff called a threshold.
#
# Lower threshold:
# - catches more true converters
# - also creates more false positives
#
# Higher threshold:
# - fewer false positives
# - but misses more real converters

thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

threshold_results = []

for threshold in thresholds:

    # Convert predicted probabilities into 0/1 predictions
    # using the current threshold.
    threshold_pred = (y_prob >= threshold).astype(int)

    precision = precision_score(
        y_test,
        threshold_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        threshold_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        threshold_pred,
        zero_division=0
    )

    predicted_positive = threshold_pred.sum()

    threshold_results.append({
        "threshold": threshold,
        "predicted_converters": predicted_positive,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_results_df = pd.DataFrame(threshold_results)

threshold_results_df

,threshold,predicted_converters,precision,recall,f1
0,0.2,12800,0.009062,1.000000,0.017962
1,0.3,12324,0.009331,0.991379,0.018489
2,0.4,9576,0.010547,0.870690,0.020842
3,0.5,5148,0.013015,0.577586,0.025456
4,0.6,1676,0.014916,0.215517,0.027902
5,0.7,278,0.017986,0.043103,0.025381
6,0.8,10,0.100000,0.008621,0.015873


In [13]:
# -------------------------
# Inspect logistic regression coefficients
# -------------------------
#
# The model has learned one coefficient for each processed feature.
#
# Positive coefficient:
# pushes predicted conversion probability upward.
#
# Negative coefficient:
# pushes predicted conversion probability downward.

# Get the fitted preprocessing step.
fitted_preprocessor = conversion_model.named_steps["preprocessor"]

# Get the final feature names after one-hot encoding.
feature_names = fitted_preprocessor.get_feature_names_out()

# Get the trained logistic regression coefficients.
coefficients = conversion_model.named_steps["model"].coef_[0]

# Put them into a readable table.
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

# Sort by absolute importance so the strongest effects appear first.
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

coef_df = coef_df.sort_values(
    "abs_coefficient",
    ascending=False
)

coef_df[["feature", "coefficient"]].head(15)

,feature,coefficient
4,num__newbie,-0.429204
11,cat__purchase_preference_Both,0.224503
5,cat__zip_code_Rural,0.215632
3,num__womens,0.202641
10,cat__channel_Web,0.148185
2,num__mens,0.139454
6,cat__zip_code_Suburban,-0.121977
9,cat__channel_Phone,-0.110384
12,cat__purchase_preference_Mens Only,-0.085049
8,cat__channel_Multichannel,0.079791


In [14]:
from sklearn.ensemble import RandomForestClassifier

# -------------------------
# Build a Random Forest baseline
# -------------------------
#
# Logistic regression learns mostly straight-line relationships.
#
# Random Forest can learn more complicated rules such as:
#
# "If history is high AND channel is Web AND recency is low..."
#
# without us manually writing those interactions.

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,

                # Helps the model pay attention to the rare
                # conversion class.
                class_weight="balanced_subsample",

                # Prevents individual trees from growing endlessly
                # and memorizing the training data too aggressively.
                max_depth=10,

                min_samples_leaf=10,

                random_state=42,

                # Use all available CPU cores.
                n_jobs=-1
            )
        )
    ]
)

# Train using the same training set as logistic regression.
random_forest_model.fit(
    X_train,
    y_train
)

print("Random Forest trained.")

Random Forest trained.


In [15]:
# -------------------------
# Evaluate Random Forest
# -------------------------
#
# We use the SAME hidden test set and SAME metrics
# as Logistic Regression so the comparison is fair.

rf_prob = random_forest_model.predict_proba(X_test)[:, 1]

rf_roc_auc = roc_auc_score(
    y_test,
    rf_prob
)

rf_pr_auc = average_precision_score(
    y_test,
    rf_prob
)

print("Logistic Regression ROC-AUC:", roc_auc)
print("Logistic Regression PR-AUC:", pr_auc)

print("\nRandom Forest ROC-AUC:", rf_roc_auc)
print("Random Forest PR-AUC:", rf_pr_auc)

Logistic Regression ROC-AUC: 0.6215201883448058
Logistic Regression PR-AUC: 0.017796938874125506

Random Forest ROC-AUC: 0.5079406991159103
Random Forest PR-AUC: 0.00964682865120408


In [16]:
# -------------------------
# Compare baseline models
# -------------------------
#
# Put all model results into one table so we can clearly see
# which model performs best on the same test set.

model_comparison = pd.DataFrame([
    {
        "model": "Dummy Classifier",
        "roc_auc": dummy_roc_auc,
        "pr_auc": dummy_pr_auc
    },
    {
        "model": "Logistic Regression",
        "roc_auc": roc_auc,
        "pr_auc": pr_auc
    },
    {
        "model": "Random Forest",
        "roc_auc": rf_roc_auc,
        "pr_auc": rf_pr_auc
    }
])

# Sort by PR-AUC because this metric is especially important
# for our highly imbalanced conversion problem.
model_comparison = model_comparison.sort_values(
    "pr_auc",
    ascending=False
)

model_comparison

,model,roc_auc,pr_auc
1,Logistic Regression,0.621520,0.017797
2,Random Forest,0.507941,0.009647
0,Dummy Classifier,0.500000,0.009062


In [17]:
# Save the model comparison so we can reuse it later
# in the README, dashboard, and final project report.

model_comparison.to_csv(
    "../data/processed/predictive_model_comparison.csv",
    index=False
)

print("Saved predictive_model_comparison.csv")

Saved predictive_model_comparison.csv


In [18]:
# -------------------------
# Prepare the spend-prediction dataset
# -------------------------
#
# We use the SAME customer features as before.
#
# The target is now "spend" instead of "conversion".
#
# Spend is a continuous number:
# $0, $10.50, $75.20, etc.
#
# Because we want a fair test, we again keep
# 20% of customers hidden from the model.

X_spend = df[feature_columns].copy()
y_spend = df["spend"].copy()

X_spend_train, X_spend_test, y_spend_train, y_spend_test = train_test_split(
    X_spend,
    y_spend,
    test_size=0.20,
    random_state=42
)

print("Training customers:", len(X_spend_train))
print("Test customers:", len(X_spend_test))

print("\nAverage training spend:", y_spend_train.mean())
print("Average test spend:", y_spend_test.mean())

print("\nCustomers with $0 spend in test set:")
print((y_spend_test == 0).sum())

print("\nCustomers with positive spend in test set:")
print((y_spend_test > 0).sum())

Training customers: 51200
Test customers: 12800

Average training spend: 1.0220763671875
Average test spend: 1.1662359375

Customers with $0 spend in test set:
12672

Customers with positive spend in test set:
128


In [19]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# -------------------------
# Build a no-skill spend baseline
# -------------------------
#
# This model does not learn customer behavior.
#
# It simply predicts the SAME average spend
# for every customer.
#
# Example:
# Customer A -> $1.02
# Customer B -> $1.02
# Customer C -> $1.02
#
# Our real regression model should ideally beat this.

dummy_spend_model = DummyRegressor(
    strategy="mean"
)

# Train the dummy model.
dummy_spend_model.fit(
    X_spend_train,
    y_spend_train
)

# Predict spend for the hidden test customers.
dummy_spend_pred = dummy_spend_model.predict(
    X_spend_test
)

# Calculate baseline metrics.
dummy_spend_mae = mean_absolute_error(
    y_spend_test,
    dummy_spend_pred
)

dummy_spend_rmse = mean_squared_error(
    y_spend_test,
    dummy_spend_pred
) ** 0.5

dummy_spend_r2 = r2_score(
    y_spend_test,
    dummy_spend_pred
)

print("Dummy Spend MAE:", dummy_spend_mae)
print("Dummy Spend RMSE:", dummy_spend_rmse)
print("Dummy Spend R²:", dummy_spend_r2)

Dummy Spend MAE: 2.1678707773437496
Dummy Spend RMSE: 15.611800661912648
Dummy Spend R²: -8.527426722060305e-05


In [20]:
from sklearn.ensemble import RandomForestRegressor

# -------------------------
# Build a spend prediction model
# -------------------------
#
# This model tries to predict the actual dollar amount
# each customer will spend.
#
# Unlike the dummy model, it can learn patterns from:
# recency, history, channel, preferences, etc.

random_forest_spend_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,

                # Limit tree depth so the model does not
                # memorize the training data too aggressively.
                max_depth=10,

                # Require at least this many customers
                # in a final tree leaf.
                min_samples_leaf=10,

                random_state=42,

                # Use all CPU cores.
                n_jobs=-1
            )
        )
    ]
)

# Train the model.
random_forest_spend_model.fit(
    X_spend_train,
    y_spend_train
)

print("Random Forest spend model trained.")

Random Forest spend model trained.


In [21]:
# -------------------------
# Evaluate Random Forest spend model
# -------------------------
#
# We compare it against the dummy baseline using
# the exact same test set and metrics.

rf_spend_pred = random_forest_spend_model.predict(
    X_spend_test
)

rf_spend_mae = mean_absolute_error(
    y_spend_test,
    rf_spend_pred
)

rf_spend_rmse = mean_squared_error(
    y_spend_test,
    rf_spend_pred
) ** 0.5

rf_spend_r2 = r2_score(
    y_spend_test,
    rf_spend_pred
)

print("Dummy Spend MAE:", dummy_spend_mae)
print("Random Forest Spend MAE:", rf_spend_mae)

print("\nDummy Spend RMSE:", dummy_spend_rmse)
print("Random Forest Spend RMSE:", rf_spend_rmse)

print("\nDummy Spend R²:", dummy_spend_r2)
print("Random Forest Spend R²:", rf_spend_r2)

Dummy Spend MAE: 2.1678707773437496
Random Forest Spend MAE: 2.1772100418031535

Dummy Spend RMSE: 15.611800661912648
Random Forest Spend RMSE: 15.668775153427177

Dummy Spend R²: -8.527426722060305e-05
Random Forest Spend R²: -0.007398117015255945


In [22]:
# -------------------------
# Inspect spend only among customers who actually purchased
# -------------------------
#
# Most customers have spend = $0.
#
# Here we temporarily look only at customers with positive spend
# so we can understand how large their purchases are.

positive_spend = df.loc[
    df["spend"] > 0,
    "spend"
]

print("Number of customers with positive spend:", len(positive_spend))

print("\nPositive spend summary:")
print(
    positive_spend.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

Number of customers with positive spend: 578

Positive spend summary:
count    578.000000
mean     116.363547
std      107.871537
min       29.990000
25%       32.272500
50%       80.795000
75%      153.350000
90%      256.488000
95%      362.200000
99%      499.000000
max      499.000000
Name: spend, dtype: float64


In [23]:
# -------------------------
# Create data for Stage 2:
# "How much does a purchaser spend?"
# -------------------------
#
# Stage 1 already predicts whether someone converts.
#
# Stage 2 should therefore only learn from customers
# who actually converted / spent money.

purchasers_df = df[df["spend"] > 0].copy()

# Use the same pre-treatment customer features.
X_purchase_amount = purchasers_df[feature_columns].copy()

# Target = dollar amount spent after purchase.
y_purchase_amount = purchasers_df["spend"].copy()

# Split purchasers into training and test sets.
X_amount_train, X_amount_test, y_amount_train, y_amount_test = train_test_split(
    X_purchase_amount,
    y_purchase_amount,
    test_size=0.20,
    random_state=42
)

print("Total purchasers:", len(purchasers_df))
print("Purchasers used for training:", len(X_amount_train))
print("Purchasers used for testing:", len(X_amount_test))

print("\nTraining average purchase amount:", y_amount_train.mean())
print("Test average purchase amount:", y_amount_test.mean())

Total purchasers: 578
Purchasers used for training: 462
Purchasers used for testing: 116

Training average purchase amount: 116.1678354978355
Test average purchase amount: 117.14301724137931


In [24]:
# -------------------------
# Stage 2 dummy baseline
# -------------------------
#
# This model predicts the SAME average purchase amount
# for every purchaser.
#
# Example:
# Buyer A -> ~$116
# Buyer B -> ~$116
# Buyer C -> ~$116
#
# A useful Stage 2 model should beat this.

dummy_amount_model = DummyRegressor(
    strategy="mean"
)

# Train only on customers who actually purchased.
dummy_amount_model.fit(
    X_amount_train,
    y_amount_train
)

# Predict purchase amounts for the hidden purchaser test set.
dummy_amount_pred = dummy_amount_model.predict(
    X_amount_test
)

# Evaluate using the same regression metrics.
dummy_amount_mae = mean_absolute_error(
    y_amount_test,
    dummy_amount_pred
)

dummy_amount_rmse = mean_squared_error(
    y_amount_test,
    dummy_amount_pred
) ** 0.5

dummy_amount_r2 = r2_score(
    y_amount_test,
    dummy_amount_pred
)

print("Dummy Purchase Amount MAE:", dummy_amount_mae)
print("Dummy Purchase Amount RMSE:", dummy_amount_rmse)
print("Dummy Purchase Amount R²:", dummy_amount_r2)

Dummy Purchase Amount MAE: 82.09847775787433
Dummy Purchase Amount RMSE: 111.00912374969016
Dummy Purchase Amount R²: -7.71768931489003e-05


In [25]:
# -------------------------
# Stage 2 real model:
# predict purchase amount among buyers
# -------------------------
#
# This model only sees customers who actually purchased.
#
# Its job is NOT to predict whether someone buys.
# Its job is:
#
# "If this person buys, how much are they likely to spend?"

purchase_amount_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,

                # Keep trees reasonably shallow to reduce overfitting.
                max_depth=8,

                # Require multiple customers in each final leaf.
                min_samples_leaf=8,

                random_state=42,

                # Use all CPU cores.
                n_jobs=-1
            )
        )
    ]
)

# Train only on purchasers.
purchase_amount_model.fit(
    X_amount_train,
    y_amount_train
)

print("Stage 2 purchase amount model trained.")

Stage 2 purchase amount model trained.


In [26]:
# -------------------------
# Evaluate Stage 2 purchase amount model
# -------------------------
#
# We compare the Random Forest against the dummy model
# using the SAME hidden purchaser test set.

amount_pred = purchase_amount_model.predict(
    X_amount_test
)

amount_mae = mean_absolute_error(
    y_amount_test,
    amount_pred
)

amount_rmse = mean_squared_error(
    y_amount_test,
    amount_pred
) ** 0.5

amount_r2 = r2_score(
    y_amount_test,
    amount_pred
)

print("Dummy Purchase Amount MAE:", dummy_amount_mae)
print("Random Forest Purchase Amount MAE:", amount_mae)

print("\nDummy Purchase Amount RMSE:", dummy_amount_rmse)
print("Random Forest Purchase Amount RMSE:", amount_rmse)

print("\nDummy Purchase Amount R²:", dummy_amount_r2)
print("Random Forest Purchase Amount R²:", amount_r2)

Dummy Purchase Amount MAE: 82.09847775787433
Random Forest Purchase Amount MAE: 85.2951772005285

Dummy Purchase Amount RMSE: 111.00912374969016
Random Forest Purchase Amount RMSE: 112.75580211306169

Dummy Purchase Amount R²: -7.71768931489003e-05
Random Forest Purchase Amount R²: -0.031796296272568636


In [27]:
# -------------------------
# Build a two-stage expected-spend baseline
# -------------------------
#
# Stage 1:
# Logistic Regression estimates:
#
#     P(customer converts)
#
# Stage 2:
# Our experiments showed that customer features do NOT
# predict purchase amount better than simply using the average.
#
# Therefore:
#
# Expected Spend
#     =
# Conversion Probability
#     ×
# Average Purchase Amount
#
# Important:
# We calculate the average purchase amount using ONLY
# customers in the training set to avoid data leakage.


# Get the original training rows using their dataframe indexes.
training_rows = df.loc[X_train.index]

# Find purchasers only inside the training data.
training_purchasers = training_rows[
    training_rows["spend"] > 0
]

# Calculate average purchase amount using training data only.
average_purchase_amount_train = training_purchasers["spend"].mean()

print(
    "Average purchase amount from training data:",
    average_purchase_amount_train
)


# -------------------------
# Predict expected spend
# -------------------------
#
# y_prob already contains Logistic Regression conversion
# probabilities for the 12,800 hidden test customers.

expected_spend_pred = (
    y_prob * average_purchase_amount_train
)


# Get the REAL spend for exactly those same test customers.
actual_test_spend = df.loc[
    X_test.index,
    "spend"
]


# Evaluate expected-spend predictions.
expected_spend_mae = mean_absolute_error(
    actual_test_spend,
    expected_spend_pred
)

expected_spend_rmse = mean_squared_error(
    actual_test_spend,
    expected_spend_pred
) ** 0.5

expected_spend_r2 = r2_score(
    actual_test_spend,
    expected_spend_pred
)


print("\nTwo-Stage Expected Spend MAE:", expected_spend_mae)
print("Two-Stage Expected Spend RMSE:", expected_spend_rmse)
print("Two-Stage Expected Spend R²:", expected_spend_r2)

Average purchase amount from training data: 117.59209956709957

Two-Stage Expected Spend MAE: 56.18391171498514
Two-Stage Expected Spend RMSE: 58.145645503946575
Two-Stage Expected Spend R²: -15.550620984470864


In [28]:
# -------------------------
# Check whether our conversion probabilities are calibrated
# -------------------------
#
# The actual conversion rate is only around 0.9%.
#
# If our model's AVERAGE predicted conversion probability
# is dramatically higher than that, then we should NOT
# use those probabilities directly for expected revenue.

print("Actual test conversion rate:")
print(y_test.mean())

print("\nAverage predicted probability:")
print(y_prob.mean())

print("\nMinimum predicted probability:")
print(y_prob.min())

print("\nMedian predicted probability:")
print(pd.Series(y_prob).median())

print("\nMaximum predicted probability:")
print(y_prob.max())

print("\nAverage predicted expected spend:")
print(expected_spend_pred.mean())

print("\nActual average test spend:")
print(actual_test_spend.mean())

Actual test conversion rate:
0.0090625

Average predicted probability:
0.47683272934748805

Minimum predicted probability:
0.23289239929781771

Median predicted probability:
0.4724748604670693

Maximum predicted probability:
0.905733615642095

Average predicted expected spend:
56.071761786281655

Actual average test spend:
1.0102015625


In [29]:
from sklearn.metrics import brier_score_loss

# -------------------------
# Train a probability-focused logistic regression
# -------------------------
#
# IMPORTANT:
#
# Earlier we used:
#     class_weight="balanced"
#
# That helped the model pay attention to rare converters,
# but it distorted the predicted probabilities.
#
# Now we train a second logistic regression WITHOUT
# class weighting.
#
# This model may have worse recall at a 0.5 threshold,
# but its predicted probabilities should be much closer
# to the real conversion rate.

probability_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

# Train on the same training data.
probability_model.fit(
    X_train,
    y_train
)

# Get predicted conversion probabilities.
probability_model_prob = probability_model.predict_proba(
    X_test
)[:, 1]

# Evaluate ranking ability.
probability_model_roc_auc = roc_auc_score(
    y_test,
    probability_model_prob
)

probability_model_pr_auc = average_precision_score(
    y_test,
    probability_model_prob
)

# Brier score measures probability accuracy.
# Lower is better.
probability_model_brier = brier_score_loss(
    y_test,
    probability_model_prob
)

print("Actual test conversion rate:")
print(y_test.mean())

print("\nAverage predicted probability:")
print(probability_model_prob.mean())

print("\nROC-AUC:")
print(probability_model_roc_auc)

print("\nPR-AUC:")
print(probability_model_pr_auc)

print("\nBrier Score:")
print(probability_model_brier)

Actual test conversion rate:
0.0090625

Average predicted probability:
0.00906057798739109

ROC-AUC:
0.620222735131961

PR-AUC:
0.017064616603162712

Brier Score:
0.008967254437353085


In [30]:
# -------------------------
# Recalculate expected spend using realistic probabilities
# -------------------------
#
# Expected Spend =
# calibrated conversion probability
# ×
# average purchase amount
#
# Example:
# 1% probability × $117 purchase amount
# ≈ $1.17 expected spend

calibrated_expected_spend_pred = (
    probability_model_prob
    * average_purchase_amount_train
)

# Evaluate against actual customer spend.
calibrated_spend_mae = mean_absolute_error(
    actual_test_spend,
    calibrated_expected_spend_pred
)

calibrated_spend_rmse = mean_squared_error(
    actual_test_spend,
    calibrated_expected_spend_pred
) ** 0.5

calibrated_spend_r2 = r2_score(
    actual_test_spend,
    calibrated_expected_spend_pred
)

print(
    "Average predicted expected spend:",
    calibrated_expected_spend_pred.mean()
)

print(
    "Actual average test spend:",
    actual_test_spend.mean()
)

print(
    "\nCalibrated Two-Stage MAE:",
    calibrated_spend_mae
)

print(
    "Calibrated Two-Stage RMSE:",
    calibrated_spend_rmse
)

print(
    "Calibrated Two-Stage R²:",
    calibrated_spend_r2
)

print(
    "\nDummy Spend MAE:",
    dummy_spend_mae
)

print(
    "Dummy Spend RMSE:",
    dummy_spend_rmse
)

print(
    "Dummy Spend R²:",
    dummy_spend_r2
)

Average predicted expected spend: 1.0654523888287637
Actual average test spend: 1.0102015625

Calibrated Two-Stage MAE: 2.0526522918956354
Calibrated Two-Stage RMSE: 14.291381435976325
Calibrated Two-Stage R²: 0.00016503058438810392

Dummy Spend MAE: 2.1678707773437496
Dummy Spend RMSE: 15.611800661912648
Dummy Spend R²: -8.527426722060305e-05


In [31]:
# -------------------------
# Fair spend comparison
# -------------------------
#
# Our calibrated two-stage model is evaluated on the customers
# in X_test.
#
# The earlier dummy spend model used a DIFFERENT random split.
#
# Therefore, we create a new dummy prediction for exactly
# the SAME customers used by the two-stage model.
#
# To avoid data leakage, the dummy prediction uses only the
# average spend from the training customers.

# Get actual spend for the classification training customers.
training_spend = df.loc[
    X_train.index,
    "spend"
]

# Calculate average spend using training data only.
average_training_spend = training_spend.mean()

# Predict that same average for every customer in X_test.
matched_dummy_spend_pred = [
    average_training_spend
] * len(X_test)

# Evaluate on the EXACT SAME actual customers.
matched_dummy_mae = mean_absolute_error(
    actual_test_spend,
    matched_dummy_spend_pred
)

matched_dummy_rmse = mean_squared_error(
    actual_test_spend,
    matched_dummy_spend_pred
) ** 0.5

matched_dummy_r2 = r2_score(
    actual_test_spend,
    matched_dummy_spend_pred
)

print("Training average spend:", average_training_spend)

print("\nMatched Dummy MAE:", matched_dummy_mae)
print("Two-Stage MAE:", calibrated_spend_mae)

print("\nMatched Dummy RMSE:", matched_dummy_rmse)
print("Two-Stage RMSE:", calibrated_spend_rmse)

print("\nMatched Dummy R²:", matched_dummy_r2)
print("Two-Stage R²:", calibrated_spend_r2)

Training average spend: 1.0610849609375002

Matched Dummy MAE: 2.0520543585205075
Two-Stage MAE: 2.0526522918956354

Matched Dummy RMSE: 14.292651414983649
Two-Stage RMSE: 14.291381435976325

Matched Dummy R²: -1.2674537481549564e-05
Two-Stage R²: 0.00016503058438810392


In [32]:
# -------------------------
# Final predictive baseline summary
# -------------------------
#
# We save the important benchmark results so we can reuse them
# later in the README, dashboard, report, and causal-model comparison.

predictive_baseline_summary = pd.DataFrame([
    {
        "task": "Conversion prediction",
        "model": "Dummy Classifier",
        "roc_auc": dummy_roc_auc,
        "pr_auc": dummy_pr_auc,
        "mae": None,
        "rmse": None,
        "r2": None
    },
    {
        "task": "Conversion prediction",
        "model": "Logistic Regression",
        "roc_auc": probability_model_roc_auc,
        "pr_auc": probability_model_pr_auc,
        "mae": None,
        "rmse": None,
        "r2": None
    },
    {
        "task": "Expected spend prediction",
        "model": "Matched Dummy",
        "roc_auc": None,
        "pr_auc": None,
        "mae": matched_dummy_mae,
        "rmse": matched_dummy_rmse,
        "r2": matched_dummy_r2
    },
    {
        "task": "Expected spend prediction",
        "model": "Two-Stage Expected Spend",
        "roc_auc": None,
        "pr_auc": None,
        "mae": calibrated_spend_mae,
        "rmse": calibrated_spend_rmse,
        "r2": calibrated_spend_r2
    }
])

predictive_baseline_summary.to_csv(
    "../data/processed/predictive_baseline_summary.csv",
    index=False
)

predictive_baseline_summary

,task,model,roc_auc,pr_auc,mae,rmse,r2
0,Conversion prediction,Dummy Classifier,0.500000,0.009062,NaN,NaN,NaN
1,Conversion prediction,Logistic Regression,0.620223,0.017065,NaN,NaN,NaN
2,Expected spend prediction,Matched Dummy,NaN,NaN,2.052054,14.292651,-0.000013
3,Expected spend prediction,Two-Stage Expected Spend,NaN,NaN,2.052652,14.291381,0.000165
